In [1]:
import laspy
import numpy as np
import CSF
import pickle
import open3d as o3d
from scipy.interpolate import griddata
from scipy import ndimage as ndi
from pathlib import Path
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import pandas as pd
import matplotlib.pyplot as plt 
from scipy.spatial import ConvexHull
from scipy.spatial import Delaunay
import alphashape
from scipy.interpolate import NearestNDInterpolator
import torch

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
def read_las(las_path):
    """
    Reads a LAS file and returns the point cloud as a NumPy array.

    :param las_path: Path to the LAS file.
    :return: NumPy array of shape (N, 3) containing the point cloud coordinates.
    """
    las = laspy.read(las_path)
    points = np.vstack((las.x, las.y, las.z)).transpose()
    return points


In [ ]:

def segment_terrain_points(points, resolution=0.5, min_tree_height=4.0):
    """
    Receives a point cloud and segments terrain points using CSF,
    then normalizes heights to create a CHM.
    """
    # DTM and height normalization
    min_x, max_x = np.min(points[:, 0]), np.max(points[:, 0])
    min_y, max_y = np.min(points[:, 1]), np.max(points[:, 1])

    # Build grid ensuring full coverage
    grid_x, grid_y = np.mgrid[min_x:max_x:resolution, min_y:max_y:resolution]

    csf = CSF.CSF()
    csf.params.bSloopSmooth = True
    csf.params.rigidness = 1
    csf.params.cloth_resolution = 0.75
    csf.params.time_step = 0.75
    csf.params.iterations = 500
    csf.setPointCloud(points)
    ground_indices = CSF.VecInt(); non_ground_indices = CSF.VecInt()
    csf.do_filtering(ground_indices, non_ground_indices)

    csf_ground_points = points[ground_indices]

    interpolated_dtm = griddata(
        points=csf_ground_points[:, :2],
        values=csf_ground_points[:, 2],
        xi=(grid_x, grid_y),
        method='linear',
    )

    if np.isnan(interpolated_dtm).any():
        nan_mask = np.isnan(interpolated_dtm)
        nan_filler = NearestNDInterpolator(csf_ground_points[:, :2], csf_ground_points[:, 2])
        nan_coords = np.vstack((grid_x[nan_mask], grid_y[nan_mask])).T
        interpolated_dtm[nan_mask] = nan_filler(nan_coords)

    # terrain_points is saved optionally; kept for reference
    terrain_points = np.vstack((grid_x.ravel(), grid_y.ravel(), interpolated_dtm.ravel())).T
    np.save("terrain_points.npy", terrain_points)

    # Safe indices (clamped)
    idx_x_all = np.clip(((points[:, 0] - min_x) / resolution).astype(int), 0, interpolated_dtm.shape[0] - 1)
    idx_y_all = np.clip(((points[:, 1] - min_y) / resolution).astype(int), 0, interpolated_dtm.shape[1] - 1)

    ground_z = interpolated_dtm[idx_x_all, idx_y_all]
    above_ground_height = points[:, 2] - ground_z

    # candidate_mask = above_ground_height > min_tree_height
    # candidate_tree_points = points[candidate_mask]
    # candidate_heights = above_ground_height[candidate_mask]
    # remaining_points = points[~candidate_mask]

    return ground_z, above_ground_height, idx_x_all, idx_y_all, terrain_points

In [ ]:
import numpy as np
import alphashape
import math

def compute_equivalent_crown_diameter(tree_points, alpha=0.15):
    """
    Computes the crown diameter (CD) from the area of a 2D Alpha Shape.

    Args:
        tree_points (np.array): Array of (N, 3) with X, Y, Z coordinates.
        alpha (float): Fitting level (0.0 = Convex Hull).

    Returns:
        float: Equivalent crown diameter (CD).
    """
    # Project to 2D (only X and Y matter for crown estimation)
    points_2d = tree_points[:, :2]

    try:
        # Compute the Alpha Shape
        hull = alphashape.alphashape(points_2d, alpha)

        if hull.is_empty:
            # If alpha is too high for point density, fall back to Convex Hull
            hull = alphashape.alphashape(points_2d, 0.0)

        # Get area in square metres
        area_2d = hull.area

        # Apply the Equivalent Diameter formula: A = pi*(CD/2)^2 => CD = 2*sqrt(A/pi)
        equivalent_cd = 2 * math.sqrt(area_2d / math.pi)

        return equivalent_cd, area_2d

    except Exception as e:
        print(f"Error in computation: {e}")
        return 0.0, 0.0

In [ ]:
import numpy as np

def compute_agb(h, cd):
    """
    Computes total Above-Ground Biomass (AGB) in kilograms.

    Args:
        h (float): Total tree height in metres.
        cd (float): Equivalent crown diameter in metres.

    Returns:
        float: Estimated biomass in kg (dry weight).
    """
    # Parameters for the formula based on the paper:
    alpha_base = 0.016
    beta_base = 2.013
    sigma_v = 0.69
    cf = np.exp((sigma_v**2) / 2)  # Log-space correction factor (~1.2687)

    alpha_g = 0.093
    beta_g = -0.223

    alpha_total = alpha_base + alpha_g
    beta_total = beta_base + beta_g

    # Apply the allometric formula: AGB = alpha * (H * CD)^beta * CF
    # H * CD is the base of the power term
    agb = alpha_total * (h * cd)**beta_total * cf

    return agb

In [7]:
def calculate_agb(trees_las_path, terrain_las_path, pred_path):
    tree_points = read_las(trees_las_path)
    terrain_points_raw = read_las(terrain_las_path)

    preds = torch.load(pred_path)

    # Build label array
    num_points = tree_points.shape[0]
    num_instances = preds['pred_masks'].shape[0]
    labels = np.full(num_points, -1, dtype=np.int32)
    # Assign unique ID to each mask
    for i in range(num_instances):
        mask = preds['pred_masks'][i].cpu().numpy().astype(bool)
        labels[mask] = i

    terrain_labels = np.full(terrain_points_raw.shape[0], -1, dtype=np.int32)

    labels = np.concatenate((labels, terrain_labels))

    all_points = np.vstack((tree_points, terrain_points_raw))

    ground_z, above_ground_height, idx_x_all, idx_y_all, terrain_points = segment_terrain_points(all_points, resolution=0.1)

    agbs = []

    for i in range(num_instances):
        mask = labels == i
        tree_pts = all_points[mask]
        tree_heights = above_ground_height[mask]

        if len(tree_pts) < 10:
            continue
        # if i == 3:
        #     graficar_control_alpha(tree_pts, alpha=0.8)

        max_height = np.max(tree_heights)
        crown_diameter, _ = compute_equivalent_crown_diameter(tree_pts, alpha=0.15)

        agb = compute_agb(max_height, crown_diameter)
        agbs.append(agb)

    return agbs


### For watershed

In [17]:
import sys

root = Path("../..").resolve()
sys.path.append(str(root))

from scripts.treeSegWatershed import classify_tree_watershed


In [ ]:
import numpy as np

def calculate_agb_watershed(trees_df):
    # Get instance IDs (labels)
    labels = trees_df['label'].values
    # Unique IDs, ignoring noise (-1 or 0 depending on preprocessing)
    instance_ids = np.unique(labels[labels >= 0])

    agbs = []

    for i in instance_ids:
        # Filter the DataFrame directly to preserve column names
        mask = labels == i
        tree_data = trees_df[mask]

        if tree_data.empty:
            continue

        # Extract tree height (H) in metres
        max_height = tree_data['Tree_Height'].max()

        # Extract points for the Alpha Shape (needs X and Y only)
        # Convert only the geometry columns to numpy
        geometry_points = tree_data[['X', 'Y']].values

        # Compute crown diameter (CD) in metres
        crown_diameter, _ = compute_equivalent_crown_diameter(geometry_points, alpha=0.15)

        # Compute AGB (kg) using the allometric formula
        # Gymnosperms: AGB = 0.109 * (H * CD)^1.790 * 1.2687
        agb = compute_agb(max_height, crown_diameter)
        agbs.append(agb)

    return agbs


In [49]:
# quick test
height = 8.5    # metres
diameter = 4.2  # metres
biomass = compute_agb(height, diameter)
print(f"Height: {height} m")
print(f"Diameter: {diameter} m")
print(f"Result: {biomass:.4f} kg")

Altura: 8.5 m
Diametro: 4.2 m
Resultado del cálculo: 83.1929 Kg


## AGB for the test set

In [9]:
import os

# Execution
base_path_preds = "../../data/Santomera/result/OACNN"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"
base_path_terrain = "../../data/Santomera/tiles/terrain"

total_agbs = []

for file in os.listdir(base_path_preds):
    if file.endswith(".pth"):
        pred_path = os.path.join(base_path_preds, file)
        las_name = file.replace("_pred.pth", ".las")
        las_path = os.path.join(base_path_gt, las_name)
        terrain_path = os.path.join(base_path_terrain, las_name)
        agbs = calculate_agb(las_path, terrain_path, pred_path)
        total_agbs.extend(agbs)

In [52]:
print(f"Average AGB: {np.mean(total_agbs):.4f} kg per tree")
print(f"Total estimated AGB for the area: {np.sum(total_agbs)/1000:.4f} Mg")

AGB promedio totales: 121.8052 Kg por árbol
AGB total estimada para el área: 8.7700 Mg


## AGB Final

In [23]:
import os
import re

# Execution
base_path_preds = "../../data/Santomera/result/FinalAGBResult"
base_path_gt1 = "../../data/Santomera/tiles/trees/labeled"
base_path_gt2 = "../../data/Santomera/tiles/trees/unlabeled"

base_path_terrain = "../../data/Santomera/tiles/terrain"

total_agbs = []

regex = r"tile_\d+"

for file in os.listdir(base_path_preds):
    if file.endswith(".pth"):
        pred_path = os.path.join(base_path_preds, file)
        las_name = file.replace("_pred.pth", ".las")
        las_path = os.path.join(base_path_gt1, las_name)
        if not os.path.exists(las_path):
            las_path = os.path.join(base_path_gt2, las_name)
        terrain_path = os.path.join(base_path_terrain, las_name)
        if las_path.endswith("_left.las") or las_path.endswith("_right.las"):
            name = re.search(regex, las_path).group(0)
            terrain_path = os.path.join(base_path_terrain, name) + ".las"
        agbs = calculate_agb(las_path, terrain_path, pred_path)
        total_agbs.extend(agbs)

In [27]:
print(f"Total estimated AGB for the area: {np.sum(total_agbs)/1000:.4f} Mg")
print(f"Average AGB: {np.mean(total_agbs):.4f} kg per tree")
print(f"AGB std dev: {np.std(total_agbs):.4f} kg per tree")
print(f"Total number of trees analysed: {len(total_agbs)}")
print(f"Min AGB: {np.min(total_agbs):.4f} kg per tree")
print(f"Max AGB: {np.max(total_agbs):.4f} kg per tree")

AGB total estimada para el área: 2058.1033 Mg
AGB promedio totales: 58.6488 Kg por árbol
Desviación estándar de AGB: 79.8373 Kg por árbol
Número total de árboles analizados: 35092
AGB mínimo: 0.0911 Kg por árbol
AGB máximo: 1636.3573 Kg por árbol


## AGB computation for the conference

### OACNN

In [8]:
import os

# Execution
base_path_preds_base = "../../data/Santomera/result/OACNN"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"
base_path_terrain = "../../data/Santomera/tiles/terrain"

models = ["_0", "_25", "_50", "_100", "_ML"]

for model in models:
    total_agbs = []
    model_path = base_path_preds_base + model
    for file in os.listdir(model_path):
        if file.endswith(".pth"):
            pred_path = os.path.join(model_path, file)
            las_name = file.replace("_pred.pth", ".las")
            las_path = os.path.join(base_path_gt, las_name)
            terrain_path = os.path.join(base_path_terrain, las_name)
            agbs = calculate_agb(las_path, terrain_path, pred_path)
            total_agbs.extend(agbs)
    print(f"Model OACNN{model}: Total AGB: {np.sum(total_agbs)/1000:.4f} Mg. Number of trees: {len(total_agbs)}")

Modelo OACNN_0: AGB total: 13.7389 Mg. Numero de árboles: 211


Modelo OACNN_25: AGB total: 13.2509 Mg. Numero de árboles: 215


Modelo OACNN_50: AGB total: 12.9826 Mg. Numero de árboles: 180


Modelo OACNN_100: AGB total: 13.1812 Mg. Numero de árboles: 170


Modelo OACNN_ML: AGB total: 12.4703 Mg. Numero de árboles: 132


### PT-v3

In [9]:
import os

# Execution
base_path_preds_base = "../../data/Santomera/result/PT-v3"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"
base_path_terrain = "../../data/Santomera/tiles/terrain"

models = ["_0", "_25", "_50", "_100", "_ML"]

for model in models:
    total_agbs = []
    model_path = base_path_preds_base + model
    for file in os.listdir(model_path):
        if file.endswith(".pth"):
            pred_path = os.path.join(model_path, file)
            las_name = file.replace("_pred.pth", ".las")
            las_path = os.path.join(base_path_gt, las_name)
            terrain_path = os.path.join(base_path_terrain, las_name)
            agbs = calculate_agb(las_path, terrain_path, pred_path)
            total_agbs.extend(agbs)
    print(f"Model PT-v3{model}: Total AGB: {np.sum(total_agbs)/1000:.4f} Mg. Number of trees: {len(total_agbs)}")

Modelo PT-v3_0: AGB total: 12.5915 Mg. Numero de árboles: 117
Modelo PT-v3_25: AGB total: 12.5333 Mg. Numero de árboles: 122


Modelo PT-v3_50: AGB total: 12.5808 Mg. Numero de árboles: 144


Modelo PT-v3_100: AGB total: 12.2935 Mg. Numero de árboles: 103


Modelo PT-v3_ML: AGB total: 10.9493 Mg. Numero de árboles: 102


### SpUNET

In [10]:
import os

# Execution
base_path_preds_base = "../../data/Santomera/result/SpUNET"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"
base_path_terrain = "../../data/Santomera/tiles/terrain"

models = ["_0", "_25", "_50", "_100", "_ML"]

for model in models:
    total_agbs = []
    model_path = base_path_preds_base + model
    for file in os.listdir(model_path):
        if file.endswith(".pth"):
            pred_path = os.path.join(model_path, file)
            las_name = file.replace("_pred.pth", ".las")
            las_path = os.path.join(base_path_gt, las_name)
            terrain_path = os.path.join(base_path_terrain, las_name)
            agbs = calculate_agb(las_path, terrain_path, pred_path)
            total_agbs.extend(agbs)
    print(f"Model SpUNET{model}: Total AGB: {np.sum(total_agbs)/1000:.4f} Mg. Number of trees: {len(total_agbs)}")

Modelo SpUNET_0: AGB total: 8.6367 Mg. Numero de árboles: 200


Modelo SpUNET_25: AGB total: 9.8096 Mg. Numero de árboles: 322


Modelo SpUNET_50: AGB total: 12.1484 Mg. Numero de árboles: 180


Modelo SpUNET_100: AGB total: 12.4867 Mg. Numero de árboles: 163


Modelo SpUNET_ML: AGB total: 15.6206 Mg. Numero de árboles: 179


### Watershed

In [23]:

test_folder = "../../data/Santomera/tests"
total_agbs = []
for file in os.listdir(test_folder):
    if file.endswith(".las"):
        trees_path = os.path.join(test_folder, file)
        terrain_path = os.path.join("../../data/Santomera/tiles/terrain", file)
        trees_df = classify_tree_watershed(trees_path, terrain_path, 0.5, 1.2, 0.05)
        if not trees_df.empty:
            agbs = calculate_agb_watershed(trees_df)
            total_agbs.extend(agbs)


print(f"Total estimated AGB for the area: {np.sum(total_agbs)/1000:.4f} Mg")
print(f"Average AGB: {np.mean(total_agbs):.4f} kg per tree")
print(f"AGB std dev: {np.std(total_agbs):.4f} kg per tree")
print(f"Total number of trees analysed: {len(total_agbs)}")

AGB total estimada para el área: 16.3456 Mg
AGB promedio totales: 259.4544 Kg por árbol
Desviación estándar de AGB: 281.8812 Kg por árbol
Número total de árboles analizados: 63
